# 16 Unified Employee Intelligence Layer
**Enterprise HR AI — Workforce Intelligence & Upskilling Platform**

### Purpose:
Combine attrition prediction risk, engagement metrics, role intelligence, skill gap analytics, and learning recommendations into a single master business-level table.


In [2]:
import os
import joblib
import pandas as pd
import numpy as np

DATA_PROCESSED = "../data/processed"
MODELS_DIR = "../models"

df_attrition = pd.read_csv(os.path.join(DATA_PROCESSED, "attrition_features_engineered.csv"))
df_gaps = pd.read_csv(os.path.join(DATA_PROCESSED, "employee_skill_gaps.csv"))
df_recs = pd.read_csv(os.path.join(DATA_PROCESSED, "employee_recommendations.csv"))

# Load production ML pipeline
pipeline = joblib.load(os.path.join(MODELS_DIR, "attrition_pipeline.joblib"))

SENSITIVE_ATTRS = ['gender', 'marital_status']
TARGET_COLS = ['attrition', 'attrition_binary']
ID_COLS = ['employee_id']
feature_cols = [c for c in df_attrition.columns if c not in SENSITIVE_ATTRS + TARGET_COLS + ID_COLS]

X_all = df_attrition[feature_cols]
attrition_probs = pipeline.predict_proba(X_all)[:, 1]

# Map to Risk Level: Low (< 0.35), Medium (0.35 - 0.65), High (>= 0.65)
risk_levels = []
for p in attrition_probs:
    if p >= 0.65:
        risk_levels.append("HIGH")
    elif p >= 0.35:
        risk_levels.append("MEDIUM")
    else:
        risk_levels.append("LOW")

df_master = pd.DataFrame({
    'employee_id': df_attrition['employee_id'],
    'department': df_attrition['department'],
    'job_role': df_attrition['job_role'],
    'monthly_income': df_attrition['monthly_income'],
    'years_at_company': df_attrition['years_at_company'],
    'attrition_probability': np.round(attrition_probs, 4),
    'attrition_risk_level': risk_levels,
    'engagement_score': np.round(df_attrition['composite_satisfaction'] * 25, 2), # Scaled 1-4 to 0-100
    'current_skills_count': df_gaps['current_skills_count'],
    'missing_skills_count': df_gaps['missing_skills_count'],
    'career_readiness_pct': df_gaps['career_readiness_pct'],
    'missing_skills': df_gaps['missing_skills_list'],
    'recommended_learning_plan': df_recs['recommended_learning_plan']
})

out_master = os.path.join(DATA_PROCESSED, "employee_intelligence_master.csv")
df_master.to_csv(out_master, index=False)

print(f"=== UNIFIED EMPLOYEE INTELLIGENCE MASTER TABLE ({len(df_master)} records) ===")
print(df_master.head(10)[['employee_id', 'department', 'job_role', 'attrition_risk_level', 'attrition_probability', 'career_readiness_pct']].to_string(index=False))
print(f"\nSaved master table to: {out_master}")


=== UNIFIED EMPLOYEE INTELLIGENCE MASTER TABLE (1470 records) ===
 employee_id             department                  job_role attrition_risk_level  attrition_probability  career_readiness_pct
           1                  Sales           Sales Executive                 HIGH                 0.6589                 47.37
           2 Research & Development        Research Scientist                  LOW                 0.1523                 50.00
           4 Research & Development     Laboratory Technician               MEDIUM                 0.5247                 45.00
           5 Research & Development        Research Scientist               MEDIUM                 0.4046                 45.45
           7 Research & Development     Laboratory Technician                  LOW                 0.2630                 45.00
           8 Research & Development     Laboratory Technician                  LOW                 0.0688                 45.00
          10 Research & Development   